In [212]:
import os
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
import joblib

In [213]:
# Load dataset
df = pd.read_csv(r'ucs_train_data.csv')

In [214]:
# Select features and target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Split data into train and test sets
random_state =2 # Define random state
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)

In [215]:
# Specify the folder to save results
save_folder = f"voting_regressor_results_{random_state}"
os.makedirs(save_folder, exist_ok=True)


In [216]:
# Define individual models
lr = LinearRegression()
dt = DecisionTreeRegressor()


In [217]:
# Create Voting Regressor
voting_regressor = VotingRegressor([('lr', lr), ('dt', dt),])

In [218]:
# Train the model
start_time = time.time()
voting_regressor.fit(X_train, y_train)
training_time = time.time() - start_time


In [219]:
# Convert to hours, minutes, and seconds
hours = training_time // 3600
minutes = (training_time % 3600) // 60
seconds = training_time % 60

# Save training time
training_time_path = os.path.join(save_folder, "training_time.txt")
with open(training_time_path, "w") as file:
    file.write(f"Training time: {hours:.0f} hours {minutes:.0f} minutes {seconds:.2f} seconds")

print(f"Training time: {hours:.0f} hours {minutes:.0f} minutes {seconds:.2f} seconds")


Training time: 0 hours 0 minutes 0.01 seconds


In [220]:
# Predictions
y_train_pred = voting_regressor.predict(X_train)
y_test_pred = voting_regressor.predict(X_test)
y_whole_pred = voting_regressor.predict(X)

In [221]:
# Create a single Excel file with multiple sheets
excel_path = os.path.join(save_folder, "results.xlsx")
with pd.ExcelWriter(excel_path) as writer:
    pd.DataFrame({"y_test": y_test, "y_test_pred": y_test_pred}).to_excel(writer, sheet_name="Test", index=False)
    pd.DataFrame({"y_train": y_train, "y_train_pred": y_train_pred}).to_excel(writer, sheet_name="Train", index=False)
    pd.DataFrame({"y_whole": y, "y_whole_pred": y_whole_pred}).to_excel(writer, sheet_name="Whole", index=False)


In [222]:

# Compute and save metrics
metrics = {
    "Train": {
        "R2": r2_score(y_train, y_train_pred),
        "RMSE": root_mean_squared_error(y_train, y_train_pred),
        "MSE": mean_squared_error(y_train, y_train_pred),
        "MAE": mean_absolute_error(y_train, y_train_pred),
        "MAPE": mean_absolute_percentage_error(y_train, y_train_pred),
        "MaxE": np.max(np.abs(y_train - y_train_pred)),
        "MinE": np.min(np.abs(y_train - y_train_pred)),
    },
    "Test": {
        "R2": r2_score(y_test, y_test_pred),
        "RMSE": root_mean_squared_error(y_test, y_test_pred),
        "MSE": mean_squared_error(y_test, y_test_pred),
        "MAE": mean_absolute_error(y_test, y_test_pred),
        "MAPE": mean_absolute_percentage_error(y_test, y_test_pred),
        "MaxE": np.max(np.abs(y_test - y_test_pred)),
        "MinE": np.min(np.abs(y_test - y_test_pred)),
    },
    "Whole": {
        "R2": r2_score(y, y_whole_pred),
        "RMSE": root_mean_squared_error(y, y_whole_pred),
        "MSE": mean_squared_error(y, y_whole_pred),
        "MAE": mean_absolute_error(y, y_whole_pred),
        "MAPE": mean_absolute_percentage_error(y, y_whole_pred),
        "MaxE": np.max(np.abs(y - y_whole_pred)),
        "MinE": np.min(np.abs(y - y_whole_pred)),
    }
}
metrics_df = pd.DataFrame(metrics)
metrics_csv_path = os.path.join(save_folder, "metrics.csv")
metrics_df.to_csv(metrics_csv_path, index=True)

# Save the trained model
model_path = os.path.join(save_folder, "voting_regressor.pkl")
joblib.dump(voting_regressor, model_path)

print("Model training and evaluation complete. Results saved.")

Model training and evaluation complete. Results saved.


In [223]:
metrics_df

,Train,Test,Whole
R2,0.963725,0.933727,0.954687
RMSE,1.614334,2.189199,1.806109
MSE,2.606074,4.792594,3.262030
MAE,1.084103,1.274605,1.141254
MAPE,0.030689,0.033977,0.031676
MaxE,8.252558,11.629853,11.629853
MinE,0.004416,0.010178,0.004416
